In [10]:
import numpy as np
import pandas as pd

In [11]:
df = pd.read_csv('./dataset/Restaurant.csv')
df.head()

,CAMIS,DBA,BORO,BUILDING,STREET,ZIPCODE,PHONE,CUISINE DESCRIPTION,INSPECTION DATE,ACTION,...,INSPECTION TYPE,Latitude,Longitude,Community Board,Council District,Census Tract,BIN,BBL,NTA,Location
0,50163224,BAKED CHEESE HAUS LTD,Manhattan,1,HERALD SQUARE,10001.0,6086301355,NaN,01/01/1900,NaN,...,NaN,40.750542,-73.987541,105.0,4.0,10900.0,1089436.0,1.008100e+09,MN17,POINT (-73.987541021756 40.750541907818)
1,50170678,D'S GRAB N GO,Brooklyn,74,RICHARDSON STREET,11211.0,6463025617,NaN,01/01/1900,NaN,...,NaN,40.718374,-73.949138,301.0,34.0,51500.0,3068058.0,3.027320e+09,BK73,POINT (-73.94913846043 40.718373646442)
2,50171263,NAYA ROCK CENTER LLC,Manhattan,30,ROCKEFELLER PLAZA,10112.0,2124612812,NaN,01/01/1900,NaN,...,NaN,40.758747,-73.978692,105.0,4.0,10400.0,1076262.0,1.012658e+09,MN17,POINT (-73.978692223615 40.758747437799)
3,50172771,AZ&G INC,Queens,133-33,39 AVENUE,11354.0,7188198818,NaN,01/01/1900,NaN,...,NaN,40.759110,-73.834161,407.0,20.0,87100.0,4000000.0,4.049728e+09,QN22,POINT (-73.834160749112 40.759109919719)
4,50165402,AREFIN'S TEA MANIA NEW YORK,Queens,98-04,101 AVENUE,11416.0,3478844480,NaN,01/01/1900,NaN,...,NaN,40.685049,-73.843194,409.0,32.0,4001.0,4439922.0,4.091050e+09,QN53,POINT (-73.843193921 40.685049473493)


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 296766 entries, 0 to 296765
Data columns (total 27 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   CAMIS                  296766 non-null  int64  
 1   DBA                    296764 non-null  object 
 2   BORO                   296766 non-null  object 
 3   BUILDING               295807 non-null  object 
 4   STREET                 296765 non-null  object 
 5   ZIPCODE                293576 non-null  float64
 6   PHONE                  296752 non-null  object 
 7   CUISINE DESCRIPTION    293485 non-null  object 
 8   INSPECTION DATE        296766 non-null  object 
 9   ACTION                 293485 non-null  object 
 10  VIOLATION CODE         291191 non-null  object 
 11  VIOLATION DESCRIPTION  291190 non-null  object 
 12  CRITICAL FLAG          296766 non-null  object 
 13  SCORE                  280188 non-null  float64
 14  GRADE                  146385 non-nu

In [13]:
df.columns

Index(['CAMIS', 'DBA', 'BORO', 'BUILDING', 'STREET', 'ZIPCODE', 'PHONE',
       'CUISINE DESCRIPTION', 'INSPECTION DATE', 'ACTION', 'VIOLATION CODE',
       'VIOLATION DESCRIPTION', 'CRITICAL FLAG', 'SCORE', 'GRADE',
       'GRADE DATE', 'RECORD DATE', 'INSPECTION TYPE', 'Latitude', 'Longitude',
       'Community Board', 'Council District', 'Census Tract', 'BIN', 'BBL',
       'NTA', 'Location'],
      dtype='object')

# Early Preprocessing

Ubah data jadi inspection level 3 tahun terakhir

In [14]:
# Parse dates
for col in ["INSPECTION DATE", "GRADE DATE", "RECORD DATE"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# Filter 3 tahun terakhir
record_date = df["RECORD DATE"].dropna().max()
start_date = record_date - pd.DateOffset(years=3)

df_recent = df[
    (df["INSPECTION DATE"].notna()) &
    (df["INSPECTION DATE"] >= start_date) &
    (df["INSPECTION DATE"] <= record_date)
].copy()

df_recent["SCORE"] = pd.to_numeric(df_recent["SCORE"], errors="coerce")
df_recent["BORO"] = df_recent["BORO"].replace("0", "Unknown")
df_recent["VIOLATION CODE"] = df_recent["VIOLATION CODE"].astype("string").str.strip()
df_recent["CRITICAL FLAG"] = df_recent["CRITICAL FLAG"].astype("string").str.strip().str.lower()

inspection_key = ["CAMIS", "INSPECTION DATE", "INSPECTION TYPE"]

# Inspection-level summary yang lebih valid
df_ins = (
    df_recent.groupby(inspection_key, as_index=False)
    .agg(
        DBA=("DBA", "first"),
        BORO=("BORO", "first"),
        BUILDING=("BUILDING", "first"),
        STREET=("STREET", "first"),
        ZIPCODE=("ZIPCODE", "first"),
        PHONE=("PHONE", "first"),
        CUISINE_DESCRIPTION=("CUISINE DESCRIPTION", "first"),
        ACTION=("ACTION", "first"),
        SCORE=("SCORE", "first"),
        GRADE=("GRADE", "first"),
        GRADE_DATE=("GRADE DATE", "first"),
        Latitude=("Latitude", "first"),
        Longitude=("Longitude", "first"),
        Community_Board=("Community Board", "first"),
        Council_District=("Council District", "first"),
        Census_Tract=("Census Tract", "first"),
        BIN=("BIN", "first"),
        BBL=("BBL", "first"),
        NTA=("NTA", "first"),
        violation_count=("VIOLATION CODE", lambda x: x.notna().sum()),
        critical_count=("CRITICAL FLAG", lambda x: (x == "critical").sum()),
        has_critical=("CRITICAL FLAG", lambda x: (x == "critical").any())
    )
)

df_ins

,CAMIS,INSPECTION DATE,INSPECTION TYPE,DBA,BORO,BUILDING,STREET,ZIPCODE,PHONE,CUISINE_DESCRIPTION,...,Longitude,Community_Board,Council_District,Census_Tract,BIN,BBL,NTA,violation_count,critical_count,has_critical
0,30075445,2023-08-01,Cycle Inspection / Initial Inspection,MORRIS PARK BAKE SHOP,Bronx,1007,MORRIS PARK AVENUE,10462.0,7188924968,Bakery Products/Desserts,...,-73.855972,211.0,13.0,25200.0,2045445.0,2.041270e+09,BX37,3,2,True
1,30075445,2023-08-22,Cycle Inspection / Re-inspection,MORRIS PARK BAKE SHOP,Bronx,1007,MORRIS PARK AVENUE,10462.0,7188924968,Bakery Products/Desserts,...,-73.855972,211.0,13.0,25200.0,2045445.0,2.041270e+09,BX37,3,1,True
2,30075445,2024-11-08,Cycle Inspection / Initial Inspection,MORRIS PARK BAKE SHOP,Bronx,1007,MORRIS PARK AVENUE,10462.0,7188924968,Bakery Products/Desserts,...,-73.855972,211.0,13.0,25200.0,2045445.0,2.041270e+09,BX37,3,1,True
3,30075445,2026-02-27,Cycle Inspection / Initial Inspection,MORRIS PARK BAKE SHOP,Bronx,1007,MORRIS PARK AVENUE,10462.0,7188924968,Bakery Products/Desserts,...,-73.855972,211.0,13.0,25200.0,2045445.0,2.041270e+09,BX37,2,1,True
4,30191841,2023-04-23,Cycle Inspection / Initial Inspection,D.J. REYNOLDS,Manhattan,351,WEST 57 STREET,10019.0,2122452912,Irish,...,-73.984310,104.0,3.0,13900.0,1026048.0,1.010480e+09,MN15,2,2,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77624,50181726,2026-02-11,Pre-permit (Non-operational) / Initial Inspection,ELECTRIC GRILL,Queens,41-18,38 STREET,11101.0,9292178365,Bakery Products/Desserts,...,-73.926419,402.0,26.0,17900.0,4003123.0,4.002160e+09,QN31,1,0,False
77625,50182075,2026-02-24,Pre-permit (Non-operational) / Initial Inspection,POPPY'S COMMISSARY,Brooklyn,189,COLUMBIA STREET,11231.0,6462637108,American,...,-74.002423,306.0,39.0,5100.0,3004081.0,3.003300e+09,BK33,0,0,False
77626,50182182,2026-02-27,Administrative Miscellaneous / Initial Inspection,AUNTIE ANNE'S,Manhattan,185,GREENWICH STREET,10007.0,5164501662,Bagels/Pretzels,...,-74.012084,101.0,1.0,1300.0,1089309.0,1.000580e+09,MN25,4,0,False
77627,50182182,2026-02-27,Pre-permit (Non-operational) / Initial Inspection,AUNTIE ANNE'S,Manhattan,185,GREENWICH STREET,10007.0,5164501662,Bagels/Pretzels,...,-74.012084,101.0,1.0,1300.0,1089309.0,1.000580e+09,MN25,4,2,True


In [15]:
df_ins.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77629 entries, 0 to 77628
Data columns (total 25 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   CAMIS                77629 non-null  int64         
 1   INSPECTION DATE      77629 non-null  datetime64[ns]
 2   INSPECTION TYPE      77629 non-null  object        
 3   DBA                  77629 non-null  object        
 4   BORO                 77629 non-null  object        
 5   BUILDING             77390 non-null  object        
 6   STREET               77629 non-null  object        
 7   ZIPCODE              76772 non-null  float64       
 8   PHONE                77626 non-null  object        
 9   CUISINE_DESCRIPTION  77629 non-null  object        
 10  ACTION               77629 non-null  object        
 11  SCORE                68860 non-null  float64       
 12  GRADE                46347 non-null  object        
 13  GRADE_DATE           44221 non-

In [16]:
df_ins.duplicated().sum()

np.int64(0)

In [17]:
df_ins.isna().sum()


CAMIS                      0
INSPECTION DATE            0
INSPECTION TYPE            0
DBA                        0
BORO                       0
BUILDING                 239
STREET                     0
ZIPCODE                  857
PHONE                      3
CUISINE_DESCRIPTION        0
ACTION                     0
SCORE                   8769
GRADE                  31282
GRADE_DATE             33408
Latitude                 386
Longitude                386
Community_Board         1234
Council_District        1230
Census_Tract            1230
BIN                     1604
BBL                      386
NTA                     1234
violation_count            0
critical_count             0
has_critical               0
dtype: int64

Drop Missing Value in Score

Drop baris yang SCORE-nya null karena SCORE adalah kolom target, tidak bisa diimputasi

In [18]:
df_ins = df_ins.dropna(subset=["SCORE"]).copy()
df_ins['SCORE'].isna().sum()

np.int64(0)

In [19]:
df_ins.columns

Index(['CAMIS', 'INSPECTION DATE', 'INSPECTION TYPE', 'DBA', 'BORO',
       'BUILDING', 'STREET', 'ZIPCODE', 'PHONE', 'CUISINE_DESCRIPTION',
       'ACTION', 'SCORE', 'GRADE', 'GRADE_DATE', 'Latitude', 'Longitude',
       'Community_Board', 'Council_District', 'Census_Tract', 'BIN', 'BBL',
       'NTA', 'violation_count', 'critical_count', 'has_critical'],
      dtype='object')

Ubah tipe data

In [20]:
df_ins['INSPECTION DATE'] = pd.to_datetime(df_ins['INSPECTION DATE'], errors="coerce")
df_ins['GRADE_DATE'] = pd.to_datetime(df_ins['GRADE_DATE'], errors="coerce")

In [21]:
df_ins["SCORE"] = pd.to_numeric(df_ins["SCORE"], errors="coerce")
df_ins["Latitude"] = pd.to_numeric(df_ins["Latitude"], errors="coerce")
df_ins["Longitude"] = pd.to_numeric(df_ins["Longitude"], errors="coerce")

In [22]:
df_ins["ZIPCODE"] = df_ins["ZIPCODE"].astype(str).str.strip().str.title().astype("category")
df_ins["BORO"] = df_ins["BORO"].str.strip().str.title().astype("category")
df_ins["CUISINE_DESCRIPTION"] = df_ins["CUISINE_DESCRIPTION"].str.strip().str.title().astype("category")
df_ins["INSPECTION TYPE"] = df_ins["INSPECTION TYPE"].str.strip().str.title().astype("category")

In [23]:
df_ins.info()

<class 'pandas.core.frame.DataFrame'>
Index: 68860 entries, 0 to 77627
Data columns (total 25 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   CAMIS                68860 non-null  int64         
 1   INSPECTION DATE      68860 non-null  datetime64[ns]
 2   INSPECTION TYPE      68860 non-null  category      
 3   DBA                  68860 non-null  object        
 4   BORO                 68860 non-null  category      
 5   BUILDING             68640 non-null  object        
 6   STREET               68860 non-null  object        
 7   ZIPCODE              68860 non-null  category      
 8   PHONE                68858 non-null  object        
 9   CUISINE_DESCRIPTION  68860 non-null  category      
 10  ACTION               68860 non-null  object        
 11  SCORE                68860 non-null  float64       
 12  GRADE                46337 non-null  object        
 13  GRADE_DATE           44219 non-null 

Pecah kolom date dan kelompokkan fitur yang dipakai

In [24]:
df_ins["inspection_year"] = df_ins["INSPECTION DATE"].dt.year
df_ins["inspection_month"] = df_ins["INSPECTION DATE"].dt.month
df_ins["inspection_quarter"] = df_ins["INSPECTION DATE"].dt.quarter
df_ins["inspection_dayofweek"] = df_ins["INSPECTION DATE"].dt.dayofweek

use_feature = ['CUISINE_DESCRIPTION', 'INSPECTION TYPE', 'BORO','ZIPCODE','Latitude','Longitude','Community_Board','inspection_year', 'inspection_month', 'inspection_quarter', 'inspection_dayofweek']

df_main = df_ins[['CAMIS','INSPECTION DATE','SCORE'] + use_feature]
df_main.head()


,CAMIS,INSPECTION DATE,SCORE,CUISINE_DESCRIPTION,INSPECTION TYPE,BORO,ZIPCODE,Latitude,Longitude,Community_Board,inspection_year,inspection_month,inspection_quarter,inspection_dayofweek
0,30075445,2023-08-01,38.0,Bakery Products/Desserts,Cycle Inspection / Initial Inspection,Bronx,10462.0,40.848231,-73.855972,211.0,2023,8,3,1
1,30075445,2023-08-22,12.0,Bakery Products/Desserts,Cycle Inspection / Re-Inspection,Bronx,10462.0,40.848231,-73.855972,211.0,2023,8,3,1
2,30075445,2024-11-08,10.0,Bakery Products/Desserts,Cycle Inspection / Initial Inspection,Bronx,10462.0,40.848231,-73.855972,211.0,2024,11,4,4
3,30075445,2026-02-27,7.0,Bakery Products/Desserts,Cycle Inspection / Initial Inspection,Bronx,10462.0,40.848231,-73.855972,211.0,2026,2,1,4
4,30191841,2023-04-23,10.0,Irish,Cycle Inspection / Initial Inspection,Manhattan,10019.0,40.767326,-73.984310,104.0,2023,4,2,6


In [25]:
df_main.info()

<class 'pandas.core.frame.DataFrame'>
Index: 68860 entries, 0 to 77627
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   CAMIS                 68860 non-null  int64         
 1   INSPECTION DATE       68860 non-null  datetime64[ns]
 2   SCORE                 68860 non-null  float64       
 3   CUISINE_DESCRIPTION   68860 non-null  category      
 4   INSPECTION TYPE       68860 non-null  category      
 5   BORO                  68860 non-null  category      
 6   ZIPCODE               68860 non-null  category      
 7   Latitude              68505 non-null  float64       
 8   Longitude             68505 non-null  float64       
 9   Community_Board       67732 non-null  float64       
 10  inspection_year       68860 non-null  int32         
 11  inspection_month      68860 non-null  int32         
 12  inspection_quarter    68860 non-null  int32         
 13  inspection_dayofweek 

Handling missing value pada fitur kategorikal

In [26]:
cat_cols = ["BORO","ZIPCODE","CUISINE_DESCRIPTION", "INSPECTION TYPE"]

for col in cat_cols:
    
    df_main[col] = df_main[col].replace(r"^\s*$", pd.NA, regex=True)
    if "Unknown" not in df_main[col].cat.categories:
        df_main[col] = df_main[col].cat.add_categories(["Unknown"])
    df_main[col] = df_main[col].fillna("Unknown")

df_main[cat_cols].isna().sum()

C:\Users\MSI SWORD 16\AppData\Local\Temp\ipykernel_32336\734245793.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_main[col] = df_main[col].replace(r"^\s*$", pd.NA, regex=True)
C:\Users\MSI SWORD 16\AppData\Local\Temp\ipykernel_32336\734245793.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_main[col] = df_main[col].fillna("Unknown")
C:\Users\MSI SWORD 16\AppData\Local\Temp\ipykernel_32336\734245793.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

BORO                   0
ZIPCODE                0
CUISINE_DESCRIPTION    0
INSPECTION TYPE        0
dtype: int64

Handling Missing Value pada Fitur Community Board

In [27]:
df_main["Community_Board"] = df_main["Community_Board"].astype(str)
df_main["Community_Board"] = df_main["Community_Board"].replace(r"^\s*$", pd.NA, regex=True)
df_main["Community_Board"] = df_main["Community_Board"].fillna("Unknown")
df_main[["Community_Board"]].isna().sum()

C:\Users\MSI SWORD 16\AppData\Local\Temp\ipykernel_32336\3247748934.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_main["Community_Board"] = df_main["Community_Board"].astype(str)
C:\Users\MSI SWORD 16\AppData\Local\Temp\ipykernel_32336\3247748934.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_main["Community_Board"] = df_main["Community_Board"].replace(r"^\s*$", pd.NA, regex=True)
C:\Users\MSI SWORD 16\AppData\Local\Temp\ipykernel_32336\3247748934.py:3: SettingWithCopyWarning: 
A value is t

Community_Board    0
dtype: int64

Handling missing value pada fitur Latitude dan Longitude

In [28]:
lat_by_camis = df_main.groupby("CAMIS")["Latitude"].transform("median")
lon_by_camis = df_main.groupby("CAMIS")["Longitude"].transform("median")

df_main["Latitude"] = df_main["Latitude"].fillna(lat_by_camis)
df_main["Longitude"] = df_main["Longitude"].fillna(lon_by_camis)

df_main["Latitude"] = df_main["Latitude"].fillna(df_main["Latitude"].median())
df_main["Longitude"] = df_main["Longitude"].fillna(df_main["Longitude"].median())

C:\Users\MSI SWORD 16\AppData\Local\Temp\ipykernel_32336\3933323246.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_main["Latitude"] = df_main["Latitude"].fillna(lat_by_camis)
C:\Users\MSI SWORD 16\AppData\Local\Temp\ipykernel_32336\3933323246.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_main["Longitude"] = df_main["Longitude"].fillna(lon_by_camis)
C:\Users\MSI SWORD 16\AppData\Local\Temp\ipykernel_32336\3933323246.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a sl

In [29]:
df_main[["Latitude", "Longitude"]].info()

<class 'pandas.core.frame.DataFrame'>
Index: 68860 entries, 0 to 77627
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Latitude   68860 non-null  float64
 1   Longitude  68860 non-null  float64
dtypes: float64(2)
memory usage: 1.6 MB


Hasil Akhir Handling Missing Value

In [30]:
df_main.isna().sum()

CAMIS                   0
INSPECTION DATE         0
SCORE                   0
CUISINE_DESCRIPTION     0
INSPECTION TYPE         0
BORO                    0
ZIPCODE                 0
Latitude                0
Longitude               0
Community_Board         0
inspection_year         0
inspection_month        0
inspection_quarter      0
inspection_dayofweek    0
dtype: int64

Buat Top-N Cuisine dan Top-N Inspection Type

In [31]:
TOP_N_CUISINE = 15
TOP_N_INSTYPE = 5

top_cuisine = df_main["CUISINE_DESCRIPTION"].value_counts().nlargest(TOP_N_CUISINE).index
df_main["CUISINE_DESCRIPTION"] = (
    df_main["CUISINE_DESCRIPTION"]
    .apply(lambda x: x if x in top_cuisine else "Other")
)

top_instype = df_main["INSPECTION TYPE"].value_counts().nlargest(TOP_N_INSTYPE).index
df_main["INSPECTION TYPE"] = (
    df_main["INSPECTION TYPE"]
    .apply(lambda x: x if x in top_instype else "Other")
)

C:\Users\MSI SWORD 16\AppData\Local\Temp\ipykernel_32336\1510512999.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_main["CUISINE_DESCRIPTION"] = (
C:\Users\MSI SWORD 16\AppData\Local\Temp\ipykernel_32336\1510512999.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_main["INSPECTION TYPE"] = (


In [32]:
print(df_main["CUISINE_DESCRIPTION"].value_counts())
print()
print(df_main["INSPECTION TYPE"].value_counts())
print(df_main["CUISINE_DESCRIPTION"].nunique())
print(df_main["INSPECTION TYPE"].nunique())

CUISINE_DESCRIPTION
Other                             18539
American                          11072
Chinese                            6221
Coffee/Tea                         5862
Pizza                              4184
Latin American                     2900
Bakery Products/Desserts           2778
Mexican                            2708
Japanese                           2364
Caribbean                          2259
Italian                            2191
Chicken                            1914
Donuts                             1701
Juice, Smoothies, Fruit Salads     1418
Spanish                            1414
Hamburgers                         1335
Name: count, dtype: int64

INSPECTION TYPE
Cycle Inspection / Initial Inspection            38133
Cycle Inspection / Re-Inspection                 14453
Pre-Permit (Operational) / Initial Inspection     9378
Other                                             3067
Pre-Permit (Operational) / Re-Inspection          2774
Cycle Inspection / Reo

In [33]:
df_main.head()

,CAMIS,INSPECTION DATE,SCORE,CUISINE_DESCRIPTION,INSPECTION TYPE,BORO,ZIPCODE,Latitude,Longitude,Community_Board,inspection_year,inspection_month,inspection_quarter,inspection_dayofweek
0,30075445,2023-08-01,38.0,Bakery Products/Desserts,Cycle Inspection / Initial Inspection,Bronx,10462.0,40.848231,-73.855972,211.0,2023,8,3,1
1,30075445,2023-08-22,12.0,Bakery Products/Desserts,Cycle Inspection / Re-Inspection,Bronx,10462.0,40.848231,-73.855972,211.0,2023,8,3,1
2,30075445,2024-11-08,10.0,Bakery Products/Desserts,Cycle Inspection / Initial Inspection,Bronx,10462.0,40.848231,-73.855972,211.0,2024,11,4,4
3,30075445,2026-02-27,7.0,Bakery Products/Desserts,Cycle Inspection / Initial Inspection,Bronx,10462.0,40.848231,-73.855972,211.0,2026,2,1,4
4,30191841,2023-04-23,10.0,Other,Cycle Inspection / Initial Inspection,Manhattan,10019.0,40.767326,-73.984310,104.0,2023,4,2,6
